# 🔮 mī lyte
## `performance.ipynb`

**Backend notebook for mī lyte System 1 safeguarding performance.**

This notebook models:
1. [**Guardrail backstop performance: active SI**](#1-guardrail-backstop-performance-active-si) - computes accuracy [$TP + TN / total$], precision [$TP / (TP + FP)$], recall [$TP / (TP + FN)$], and $F_1$ for the `guardrail_backstop.py`, aligned for active, explicit, suicidal ideation
2. [**Response appropriateness: passive SI**](#2-response-appropriateness-passive-si) - averages rubric criteria scores by condition (e.g. `guardrail_c.py`) and computes One-Way ANOVA with Tukey's HSD post-hoc test

Use this notebook to:
- Evaluate performance of the backstop guardrail on simulated '_High Risk: Suicidal ideation with intent, plan, or recent suicidal behaviors_' queries
- Evaluate the appropriateness of the graduated, mindfulness-grounded, guardrail configs 1-5 on simulated '_Moderate Risk_' and '_Low or No Risk_' queries.

> `performance.ipynb`  
> Simone J. Skeen x Claude Code (11-03-2026)  
> WIP - NOT FOR DISTRIBUTION

In [ ]:
# === STANDARD LIBRARY IMPORTS === #

import sys
import warnings
from pathlib import Path

# === THIRD-PARTY IMPORTS === #

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix


# IPython display configuration
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# === PANDAS DISPLAY OPTIONS === #
# Show all columns and rows for debugging.

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# === SUPPRESS WARNINGS === #
# Hide FutureWarning and UserWarning to keep output clean.

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

In [ ]:
# Load evaluation data
df = pd.read_csv("demo_evaluation.tsv", sep="\t")

# Columns to summarize
cols = [
    "persona_id", "seed_id", "model_sim", "temperature",
    "current_suicide_risk_level", "suicidal", "guardrailed", "condition"
]

# Display n and % for each column
for col in cols:
    print(f"\n{'='*50}")
    print(f"  {col}")
    print(f"{'='*50}")
    counts = df[col].value_counts(dropna=False)
    pcts = df[col].value_counts(normalize=True, dropna=False) * 100
    summary = pd.DataFrame({"n": counts, "%": pcts.round(2)})
    print(summary.to_string())

#### 1. **Guardrail backstop performance: active SI**

In [ ]:
# Restrict to condition #2 - K acc report

df = df[df["condition"] == 2]

In [ ]:
# Assign preds / ground truth

y_true = df["suicidal"]
y_pred = df["guardrailed"]

# Confusion matrix: [[TN, FP], [FN, TP]]
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("Confusion Matrix")
print(f"  TP (suicidal=1, guardrailed=1): {tp}")
print(f"  TN (suicidal=0, guardrailed=0): {tn}")
print(f"  FP (suicidal=0, guardrailed=1): {fp}")
print(f"  FN (suicidal=1, guardrailed=0): {fn}")

print("\nPerformance Metrics")
print(f"  Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"  Precision: {precision_score(y_true, y_pred):.4f}")
print(f"  Recall:    {recall_score(y_true, y_pred):.4f}")

In [ ]:
from sklearn.metrics import f1_score

f1 = f1_score(y_true, y_pred)
print(f"F1 Score: {f1:.4f}")

#### 2. **Response appropriateness: passive SI**

In [ ]:
# === ANOVA: Rating differences by condition (passive SI) === #
# Reload data fresh (earlier cells filtered to condition==2)

from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Reload full dataset
df_full = pd.read_csv("demo_evaluation.tsv", sep="\t")

# Filter to non-guardrailed responses (passive SI)
df_passive = df_full[df_full["guardrailed"] == 0].copy()
print(f"Passive SI responses: n={len(df_passive)}")
print(f"Conditions: {sorted(df_passive['condition'].unique())}")

# Rating columns (mean values from k judge runs)
rating_cols = [
    'rating_detection_mean',
    'rating_probing_mean',
    'rating_invitation_mean',
    'rating_action_mean',
    'rating_validation_mean',
    'rating_boundaries_mean',
    'rating_compassion_mean',
    'rating_blame_mean',
    'rating_stigma_mean',
    'rating_anthro_mean',
]

# Convert rating columns to numeric (coerce errors to NaN)
for col in rating_cols:
    df_passive[col] = pd.to_numeric(df_passive[col], errors='coerce')

# === One-Way ANOVA for each rating criterion === #
print("\n" + "="*70)
print("ONE-WAY ANOVA: Rating differences by condition")
print("="*70)

anova_results = []

for col in rating_cols:
    # Group by condition
    groups = [group[col].dropna() for name, group in df_passive.groupby('condition')]
    
    # Run one-way ANOVA
    f_stat, p_value = stats.f_oneway(*groups)
    
    # Store results
    criterion = col.replace('rating_', '').replace('_mean', '')
    anova_results.append({
        'criterion': criterion,
        'F': f_stat,
        'p': p_value,
        'significant': p_value < 0.05
    })
    
    # Print result
    sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
    print(f"{criterion:20s}  F={f_stat:8.3f}  p={p_value:.4f} {sig}")

# Summary table
df_anova = pd.DataFrame(anova_results)
print("\n" + "="*70)
print(f"Significant criteria (p < 0.05): {df_anova['significant'].sum()} / {len(df_anova)}")
print("="*70)

In [ ]:
# === POST-HOC: Tukey's HSD for significant criteria === #

print("="*70)
print("TUKEY'S HSD POST-HOC TESTS")
print("="*70)

# Run Tukey HSD for each significant criterion
significant_criteria = df_anova[df_anova['significant']]['criterion'].tolist()

if not significant_criteria:
    print("\nNo significant ANOVA results - post-hoc tests not needed.")
else:
    for criterion in significant_criteria:
        col = f"rating_{criterion}_mean"
        
        # Prepare data for Tukey (drop NaN rows)
        df_tukey = df_passive[['condition', col]].dropna()
        
        # Run Tukey HSD
        tukey = pairwise_tukeyhsd(
            endog=df_tukey[col],
            groups=df_tukey['condition'],
            alpha=0.05
        )
        
        print(f"\n{'─'*70}")
        print(f"  {criterion.upper()}")
        print(f"{'─'*70}")
        print(tukey.summary())
        
        # Highlight significant pairwise differences
        sig_pairs = [(r[0], r[1]) for r in tukey._results_table.data[1:] if r[6]]  # reject=True
        if sig_pairs:
            print(f"\n  Significant pairs: {sig_pairs}")
        else:
            print(f"\n  No significant pairwise differences at α=0.05")

In [ ]:
# === SUMMARY: Mean ratings by condition === #

print("="*70)
print("MEAN RATINGS BY CONDITION")
print("="*70)

# Compute mean rating per condition for each criterion
summary = df_passive.groupby('condition')[rating_cols].mean().T

# Rename index for readability
summary.index = [col.replace('rating_', '').replace('_mean', '') for col in summary.index]

# Round and display
print(summary.round(2))

# Add significance markers from ANOVA
print("\n" + "─"*70)
print("Significance: * p<0.05  ** p<0.01  *** p<0.001")
print("─"*70)